# 01 — Exploratory Data Analysis

Understand the network-flow data before modelling it: what the flows represent,
how imbalanced the classes are, which features separate attacks from benign
traffic, and which features are redundant.

**Data source.** This notebook reads whatever is in `data/processed/`. If you
have not downloaded CIC-IDS2017, it falls back to the bundled **synthetic**
sample. Charts built on synthetic data show that the analysis code works; they
say nothing about real network traffic. The notebook prints which source it
used, and you should read every conclusion in that light.

In [ ]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import PATHS, ATTACK_CLASSES, BENIGN_LABEL
from src.data_loader import load_sample_data, normalize_columns
from src.preprocessing import clean_dataset, add_derived_preview

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (11, 5)

In [ ]:
# Prefer the processed real dataset; fall back to the synthetic sample.
processed = PATHS.processed_dataset
if processed.exists():
    df = pd.read_parquet(processed)
    SOURCE = "processed CIC-IDS2017"
elif processed.with_suffix(".csv").exists():
    df = pd.read_csv(processed.with_suffix(".csv"))
    SOURCE = "processed CIC-IDS2017 (csv)"
else:
    df, _ = clean_dataset(normalize_columns(load_sample_data()), min_samples_per_class=10)
    SOURCE = "SYNTHETIC sample"

print(f"Data source : {SOURCE}")
print(f"Shape       : {df.shape[0]:,} rows x {df.shape[1]} columns")
if "SYNTHETIC" in SOURCE:
    print("\n*** Synthetic data. Distributions are generator artefacts, not real traffic. ***")

## 1. Dataset overview

Row count, feature count, dtypes, missing values and duplicates. In flow data,
duplicates matter more than usual: a scanner emits thousands of near-identical
probes, so exact duplicates are common and would otherwise inflate the apparent
dataset size and leak between train and test splits.

In [ ]:
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique": df.nunique(),
})
display(overview.head(30))

print(f"Duplicate rows       : {df.duplicated().sum():,}")
print(f"Columns with any NaN : {int((df.isna().sum() > 0).sum())}")
print(f"Constant columns     : {[c for c in df.columns if df[c].nunique(dropna=True) <= 1]}")

In [ ]:
numeric = df.select_dtypes(include=[np.number])
display(numeric.describe().T[["mean", "std", "min", "50%", "max"]].round(2).head(25))

## 2. Class distribution and imbalance

Class imbalance is the defining statistical problem of intrusion detection.
Benign traffic dominates by orders of magnitude, so a model that predicts
"BENIGN" for everything can score very high accuracy while detecting nothing.

This is why the project selects models on **macro-F1** rather than accuracy:
macro-F1 averages the per-class F1 scores with equal weight, so a rare attack
class cannot be ignored the way it can under accuracy.

In [ ]:
counts = df["label"].value_counts()
pct = (counts / counts.sum() * 100).round(2)
dist = pd.DataFrame({"count": counts, "percent": pct})
display(dist)

benign_share = pct.get(BENIGN_LABEL, 0)
rarest = counts.idxmin()
print(f"Benign share        : {benign_share:.2f}%")
print(f"Rarest class        : {rarest} ({counts.min():,} rows)")
print(f"Imbalance ratio     : {counts.max() / counts.min():.1f} : 1")
print(f"\nAccuracy of an always-BENIGN classifier: {benign_share:.2f}%  <- the number to beat")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
counts.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Flows per class (linear)"); axes[0].set_ylabel("Flows")
counts.plot(kind="bar", ax=axes[1], color="indianred", logy=True)
axes[1].set_title("Flows per class (log scale)"); axes[1].set_ylabel("Flows (log)")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

## 3. Network behaviour by class

The question that matters for modelling: **do attack classes actually look
different in feature space?** If the distributions overlap completely, no
amount of model tuning will help.

Rate features (`flow_packets_s`, `flow_bytes_s`) should separate volumetric
floods. Directional asymmetry should separate scans. Packet-size features
should separate payload-carrying sessions from control-only traffic.

In [ ]:
enriched = add_derived_preview(df)

behaviour_features = [
    "flow_duration", "flow_packets_s", "flow_bytes_s",
    "packets_per_second", "fwd_bwd_packet_ratio", "tcp_flag_density",
]
available = [f for f in behaviour_features if f in enriched.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, feature in zip(axes.ravel(), available):
    data = enriched[[feature, "label"]].copy()
    # log1p because these features span many orders of magnitude
    data[feature] = np.log1p(data[feature].clip(lower=0))
    sns.boxplot(data=data, x="label", y=feature, ax=ax, showfliers=False)
    ax.set_title(f"log1p({feature})"); ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

In [ ]:
# Protocol and destination-port behaviour
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

if "port_category" in enriched.columns:
    ct = pd.crosstab(enriched["port_category"], enriched["label"], normalize="index")
    sns.heatmap(ct, annot=False, cmap="Blues", ax=axes[0])
    axes[0].set_title("Class mix by port category (row-normalised)")

top_ports = df["destination_port"].value_counts().head(15)
top_ports.plot(kind="barh", ax=axes[1], color="darkseagreen")
axes[1].set_title("15 most-targeted destination ports"); axes[1].set_xlabel("Flows")
plt.tight_layout(); plt.show()

## 4. Correlation analysis

Flow features are heavily redundant by construction: `average_packet_size` is
essentially total bytes divided by total packets, and several length statistics
are computed from the same underlying packet series.

**Does redundancy need removing?** It depends on the model:

- **Tree ensembles** (Random Forest, XGBoost) are largely unaffected. They pick
  one of a correlated group and ignore the rest. The cost is diluted feature
  importances, not degraded accuracy.
- **Logistic Regression** is affected. Collinearity inflates coefficient
  variance and makes individual coefficients uninterpretable, though it does
  not necessarily hurt predictive accuracy.

This project keeps the correlated features and relies on regularisation for the
linear baseline, because dropping them would cost real signal for the tree
models that ultimately win selection. The pairs below are worth knowing about
when interpreting SHAP values — attribution spreads across correlated features.

In [ ]:
corr = enriched.select_dtypes(include=[np.number]).corr(numeric_only=True)

plt.figure(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap="coolwarm", center=0, square=True,
            linewidths=0.3, cbar_kws={"shrink": 0.6})
plt.title("Feature correlation (lower triangle)"); plt.tight_layout(); plt.show()

In [ ]:
# Highly correlated pairs, ranked
pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack().reset_index()
)
pairs.columns = ["feature_a", "feature_b", "correlation"]
strong = pairs[pairs["correlation"].abs() > 0.90].sort_values(
    "correlation", key=abs, ascending=False
)
print(f"{len(strong)} feature pairs with |r| > 0.90\n")
display(strong.head(20).round(3))

## 5. Takeaways for modelling

1. **Imbalance is severe.** Accuracy is not a usable selection metric; macro-F1
   and per-class recall are. Class weighting is applied during training and the
   majority class is downsampled in the training split only.
2. **Rate and asymmetry features carry the signal.** This motivates the derived
   features in `src/feature_engineering.py` rather than relying on raw counts.
3. **Redundancy is high but tolerable.** Kept for the tree models; interpret
   SHAP values with correlated groups in mind.
4. **Identifier columns must be dropped.** Source/destination IP, flow ID and
   timestamp identify *this capture's lab topology*. Keeping them yields
   near-perfect scores that collapse on any other network — the classic leakage
   failure in published NIDS results.

Next: `02_model_experiments.ipynb`.